In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/olist_delivery_time_data.csv")

In [20]:
df

,order_id,purchase_dayofweek,purchase_month,purchase_hour,purchase_is_weekend,estimated_delivery_days,n_items,n_sellers,total_price,total_freight,...,product_width_cm,payment_installments,payment_value,payment_type,customer_state,seller_state,product_category_name_english,distance_km,same_state,delivery_time_days
0,e481f51cbdc54678b7cc49136f2d6af7,0,10,10,0,15.544063,1,1,29.99,8.72,...,13.0,1.0,38.71,voucher,SP,SP,housewares,18.681711,1,8.436574
1,53cdb2fc8bc7dce0b6741e2150273451,1,7,20,0,19.137766,1,1,118.70,22.76,...,19.0,1.0,141.46,boleto,BA,SP,perfumery,861.035367,0,13.782037
2,47770eb9100c2d0c44946d9cf07ec65d,2,8,8,0,26.639711,1,1,159.90,19.22,...,21.0,3.0,179.12,credit_card,GO,SP,auto,514.547140,0,9.394213
3,949d5b44dbf5de918fe9c16f97b45f8a,5,11,19,1,26.188819,1,1,45.00,27.20,...,20.0,1.0,72.20,credit_card,RN,MG,pet_shop,1821.802656,0,13.208750
4,ad21c59c0840e6cb83a9ceb5573f8159,1,2,21,0,12.112049,1,1,19.90,8.72,...,15.0,1.0,28.62,credit_card,SP,SP,stationery,29.593095,1,2.873877
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96473,9c5dedf39a927c1b2549525ed64a053c,3,3,9,0,18.587442,1,1,72.00,13.08,...,18.0,3.0,85.08,credit_card,SP,SP,health_beauty,69.472869,1,8.218009
96474,63943bddc261676b46f01ca7ac2f7bd8,1,2,12,0,23.459051,1,1,174.90,20.10,...,40.0,3.0,195.00,credit_card,SP,SP,baby,474.099082,1,22.193727
96475,83c1379a015df1e13d02aae0204711ab,6,8,14,1,30.384225,1,1,205.99,65.02,...,22.0,5.0,271.01,credit_card,BA,SP,home_appliances_2,967.887943,0,24.859421
96476,11c177c8e97725db2631073c19f07b62,0,1,21,0,37.105243,2,1,359.98,81.18,...,20.0,4.0,441.16,credit_card,RJ,MG,computers_accessories,370.733547,0,17.086424


In [3]:
df.shape

(96478, 24)

In [4]:
df.isnull().sum()

order_id                            0
purchase_dayofweek                  0
purchase_month                      0
purchase_hour                       0
purchase_is_weekend                 0
estimated_delivery_days             0
n_items                             0
n_sellers                           0
total_price                         0
total_freight                       0
avg_price                           0
product_weight_g                   16
product_length_cm                  16
product_height_cm                  16
product_width_cm                   16
payment_installments                1
payment_value                       1
payment_type                        1
customer_state                      0
seller_state                        0
product_category_name_english    1378
distance_km                       478
same_state                          0
delivery_time_days                  8
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.drop_duplicates(inplace=True)

In [8]:
num_cols = ['product_weight_g',
            'product_length_cm',
            'product_height_cm',
            'product_width_cm',
            'payment_installments',
            'payment_value',
            'distance_km',
            'delivery_time_days']
for col in num_cols:
    df[col].fillna(df[col].median())

In [11]:
df['payment_type'].fillna(
    df['payment_type'].mode()[0])

0            voucher
1             boleto
2        credit_card
3        credit_card
4        credit_card
            ...     
96473    credit_card
96474    credit_card
96475    credit_card
96476    credit_card
96477     debit_card
Name: payment_type, Length: 96478, dtype: object

In [16]:
df['product_category_name_english'] = df['product_category_name_english'].fillna('Unknown')

In [18]:
df.isnull().sum()

order_id                         0
purchase_dayofweek               0
purchase_month                   0
purchase_hour                    0
purchase_is_weekend              0
estimated_delivery_days          0
n_items                          0
n_sellers                        0
total_price                      0
total_freight                    0
avg_price                        0
product_weight_g                 0
product_length_cm                0
product_height_cm                0
product_width_cm                 0
payment_installments             0
payment_value                    0
payment_type                     0
customer_state                   0
seller_state                     0
product_category_name_english    0
distance_km                      0
same_state                       0
delivery_time_days               0
dtype: int64

In [29]:
Q1 = df['delivery_time_days'].quantile(0.25)
Q3 = df['delivery_time_days'].quantile(0.75)

IQR = Q3 -Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['delivery_time_days'] < lower_bound) |
              (df['delivery_time_days'] > upper_bound)]

print(outliers.shape)

(4896, 24)


In [45]:
df_clean = df[
    (df['delivery_time_days'] >= lower_bound) &
    (df['delivery_time_days'] <= upper_bound)].copy()

In [34]:
df_clean.shape

(91582, 24)

In [42]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
cols = ['distance_km',
        'total_price',
        'total_freight',
        'payment_value']
df_clean[cols] = scaler.fit_transform(df_clean[cols])

df_clean[cols].head()

,distance_km,total_price,total_freight,payment_value
0,0.002153,0.002168,0.004858,0.002133
1,0.099222,0.008769,0.012680,0.009658
2,0.059294,0.011835,0.010708,0.012416
3,0.209937,0.003285,0.015154,0.004585
4,0.003410,0.001418,0.004858,0.001394


In [43]:
df_clean.to_csv("../data/cleaned_delivery_data.csv", index=False)

In [44]:
df_clean.shape

(91582, 24)